**Note: this notebook requires the "tiler" optional dependency (`jupytergis[tiler]`).** It will not work in JupyterLite because the "tiler" optional dependency exposes an HTTP endpoint through `jupyter-server`.

# Cloud Native Geospatial (2026) demo

Goals:

* Show a Google Earth Engine (GEE)-like user experience of interactively exploring the results of a calculation done on a cloud optimized datacube.
* ?

## Environment and settings

As always, we need to install dependencies first.

We also set some environment variable to easily acces Google Cloud Storage anonymously.

In [ ]:
!pip install gcsfs dask[diagnostics] distributed

In [ ]:
import os


def set_env():
    os.environ["GS_NO_SIGN_REQUEST"] = "YES"


set_env()

In [ ]:
# NDSI > 0.4 indicates presence of snow
SNOW_DETECTION_NDSI_THRESHOLD = 0.4

TILESERVER_BASEMAP = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"

FILE_PATH_GREEN_BAND = "gs://supaero/31TCH/SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2/SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2_FRE_B3.tif"
FILE_PATH_SWIR_BAND = "gs://supaero/31TCH/SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2/SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2_FRE_B11.tif"

# We'll grab cells 4000-5000 on both axes as our area of interest
CELL_MIN = 4000
CELL_MAX = 5000

NODATA_VALUE = -10_000

## The data

We'll use Sentinel-2 L2A products.
[Learn more about these products here](https://www.theia-land.fr/en/blog/product/sentinel-2-surface-reflectance/).

## Check we have access to GCS bucket

In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem(bucket_name="supaero", token="anon")  # noqa: S106

In [ ]:
fs.ls("supaero/31TCH")

In [ ]:
fs.ls("supaero/31TCH/SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2")

## Building an Xarray `Dataset` for a single date

In [ ]:
%%time

import rioxarray
import xarray as xr

# Open short-wave infrared band dataset -- fast!
swir = rioxarray.open_rasterio(FILE_PATH_SWIR_BAND)
swir = swir[:, CELL_MIN:CELL_MAX, CELL_MIN:CELL_MAX]
swir

In [ ]:
type(swir.variable._data)  # Shows `MemoryCachedArray`, meaning not in memory right now

In [ ]:
%%time

# Fixup NODATA values to -10_000 -- loads dataset in memory
swir = swir.where(swir != NODATA_VALUE)
swir.rio.write_nodata(NODATA_VALUE, encoded=True, inplace=True)

In [ ]:
type(swir.variable._data)  # Shows numpy.ndarray, meaning in-memory

In [ ]:
%%time

# Open green band dataset -- fast!
green = rioxarray.open_rasterio(FILE_PATH_GREEN_BAND)
green = green[
    :,
    CELL_MIN * 2 : CELL_MAX * 2,
    CELL_MIN * 2 : CELL_MAX * 2,
]  # Green band is twice the resolution of SWIR band
green

In [ ]:
%%time

# Divide resolution by 2 -- loads dataset in memory. Takes ~10 seconds!
green = green.coarsen(x=2, y=2, boundary="pad").mean()

# Fixup NODATA values to -10_000
green = green.where(green != NODATA_VALUE)
green.rio.write_nodata(NODATA_VALUE, encoded=True, inplace=True)

(TODO: How many of the above calculations can be deferred?)

In [ ]:
%%time

# Combine into one dataset including deferred calculations -- FAST!
ndsi = (green - swir) / (green + swir)
ds = xr.Dataset(
    {
        "green": green,
        "swir": swir,
        "ndsi": ndsi,
        "snow": ndsi > SNOW_DETECTION_NDSI_THRESHOLD,
    },
)
ds

In [ ]:
type(ds.snow.variable._data)

In [ ]:
%%time

# This plots really fast because our datasets are in memory at this point
ds.snow.plot()

In [ ]:
float(ds.snow.sum() / ds.snow.count())

On this particular patch, you should get over 77% of snow cover.

### Visualize interactively

We can use JupyterGIS to visualize this `DataArray` interactively.
First, let's show that the data in question is loaded in memory:

In [ ]:
type(ds.ndsi.variable._data)

This isn't super exciting.
We're just interactively visualizing a dataset that fits easily in memory:

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(
    longitude=1.7092461496028497,
    latitude=42.530360648396055,
    zoom=11.38992197518327,
)
doc.add_raster_layer(url=TILESERVER_BASEMAP)
await doc.ready()
await doc.add_data_array_layer(
    name="Snow (yes/no)",
    data_array=ds.ndsi,
    colormap_name="viridis",
    colormap_range=(0, 1),
)
doc

## Compute a time series analysis (WIP)

Now that we know how to create a 2D `Dataset` for a single date, we'll build a dataset with a time dimension.
The idea is to stack temporal slices into a single `Dataset` using a new time dimension.

Up to now, we've only built datasets that easily fit in memory, by taking only part of one observation.
In order to be able to work on full images and on ten products, we'll need to use Dask.

### Start a Dask cluster

Configure it according to your computing power.

In [ ]:
from distributed import Client

client = Client(n_workers=8, threads_per_worker=2, memory_limit="4GiB")
client

Then we'll set the same environment variable as earlier to allow anonymous access to GCFS for every worker:

In [ ]:
client.run(set_env)

It's interesting when using Dask to start a real Distributed cluster (even on one machine like we've done above).
This gives access to a nice Dashboard, just click on the link displayed above (ending with 8787/status).

If you are using binder, you should replace the URL with something like:

```
https://hub.2i2c.mybinder.org/user/guillaumeeb-supaero-otsu-course-4kfqus3j/proxy/8787/status
```

First part of the URL should be copied from your Jupyterlab URL on your browser.

### Define some functions to build Datasets

In order to simplify the building of our time series, we'll define some functions.

#### Read one band of a file

It should also handle an optional resampling. This time, we also want to use a Dask backend.

- `product` is the name of the product, i.e. the `"SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2"` part, which is repeated in a folder, and in the file name.
- `band` is the band identifier, i.e. B3 or B11 in our case.
  We always access `"FRE"` bands in our case.
- `coarsen` is the level of subsambling, it should be 1 by default (no subsampling) for the SWIR band.
  Set it to 2 for GREEN to perform a resampling.
- When opening the file with rioxarray, we'll use two new arguments: `chunks`, and `lock=False`.
  `chunks` indicate that we want chunked array as a backend, so Dask Arrays.
- We also want to remove the band dimension.
  In order to do so, we apply `.squeeze('band', drop=True)` after the opening.

In [ ]:
def read_one_band(product, band, coarsen=1):
    chunks = {
        "band": -1,
        "y": 1024 * coarsen,
        "x": 1024 * coarsen,
    }
    band = rioxarray.open_rasterio(
        f"gs://supaero/31TCH/{product}/{product}_FRE_{band}.tif",
        chunks=chunks,
        lock=False,
    ).squeeze("band", drop=True)
    band = band.where(band != -10000)
    band.rio.write_nodata(-10000, encoded=True, inplace=True)
    if coarsen > 1:
        band = band.coarsen(x=coarsen, y=coarsen, boundary="pad").mean()
    return band

Try this function on the green band of "SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2", so with a coarsen=2 value, and whatch the HTML repr. What do you notice?

In [ ]:
read_one_band("SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2", "B3", 2)

#### Create a single time slice Dataset with green and swir band

`product` argument is the same as above, and it will create a 2 dimension dataset with two DataArray variables: `green` and `swir`, at the same resolution.

In [ ]:
def create_dataset(product):
    ds = xr.Dataset(
        {
            "green": read_one_band(product, "B3", 2),
            "swir": read_one_band(product, "B11"),
        },
    )
    return ds

Try it on the above product:

In [ ]:
create_dataset("SENTINEL2B_20191224-104910-788_L2A_T31TCH_C_V2-2")

### Build our time series

We are now ready to build a time series of our ten products.

In order to do that, we'll need:
- our product list
- A Pandas `DatetimeIndex` corresponding to our product list
- A list of all of our datasets
- Apply xarray `concat` on our dataset list associated with our new index as a dimension.

In [ ]:
product_list = [path.split("/")[-1] for path in fs.ls("supaero/31TCH")]
product_list

In [ ]:
import pandas as pd

# Create time index
dates = [product.split("_")[1] for product in product_list]
dt_index = pd.to_datetime(dates, format="%Y%m%d-%H%M%S-%f")
dt_index.name = "time"
dt_index

Now we create a `Dataset` from each of the products above.
Then, use `xr.concat` to create a new timeseries `Dataset` with a new time dimension and all of our single time step datasets in it.

Note the `time` dimension that's present on the new dataset.

In [ ]:
datasets = [create_dataset(product) for product in product_list]

timeseries_ds = xr.concat(datasets, dt_index)
timeseries_ds

### Add NDSI and snow mask

Just as with a single time step Dataset, you can easily add two new variables to this complete `Dataset`.

So create the new `ndsi` and `snow` variables just like we did before.

In [ ]:
timeseries_ds["ndsi"] = (timeseries_ds.green - timeseries_ds.swir) / (
    timeseries_ds.green + timeseries_ds.swir
)
timeseries_ds["snow"] = timeseries_ds.ndsi > 0.4
timeseries_ds

**The `ndsi` and `snow` variables are _not_ loaded in to memory**:

In [ ]:
type(
    timeseries_ds.snow.variable._data,
)  # A `dask.array.core.Array` is a data object with deferred computation

In [ ]:
type(timeseries_ds.ndsi.variable._data)

### Now let's visualize it with JupyterGIS and jupyter-tiler!

This is more interesting -- the data is not loaded in to memory so computations are dynamic for only the needed tiles!

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(
    longitude=1.7092461496028497,
    latitude=42.530360648396055,
    zoom=11.38992197518327,
)
doc.add_raster_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
)
await doc.ready()
doc

Now we'll add the data layer.

It may take a minute or two before data starts showing up on the map, as we're computing the values on the fly.

But because we're computing on the fly, we don't need to compute the whole dataset ahead of time.
You can zoom in and out to trigger finer or coarser computations.

In [ ]:
await doc.add_data_array_layer(
    name="NDSI Layer",
    data_array=timeseries_ds.ndsi.isel(time=0, drop=True),
    colormap_name="viridis",
    colormap_range=(-1, 1),
)

TODO: Deal with or explain "invalid value encountered in divide"

TODO: Debug why basemap tile loads are slow when also computing data tiles. Those should take precedence.

TODO: Tiler performance extremely poor when zooming out. I think we're queueing large numbers of tile requests that aren't being canceled/aborted when the zoom level changes such that those requests are no longer needed.